# NYC MTA bus speeds

This is an analysis of NYC bus speeds by route using MTA open data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
url = "https://data.ny.gov/api/views/cudb-vcni/rows.csv?accessType=DOWNLOAD"
df = pd.read_csv(url)

In [ ]:
df.head()
df.shape
df.dtypes
df.isnull().sum()

**Which routes have the lowest average speed overall?**

In [ ]:
route_speeds = df.groupby ("route_id")["average_speed"].mean()
route_speeds.sort_values().head(10)
regular_routes = df[df["route_id"].str.contains("SHTL") == False]
route_speeds_clean = regular_routes.groupby("route_id")["average_speed"].mean()
slowest = route_speeds_clean.sort_values().head(10)
slowest.plot(kind="bar")
plt.title("10 Slowest NYC Bus Routes by Average Speed")
plt.xlabel("Route")
plt.ylabel("Average Speed (mph)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Does speed change by Borough?**

In [ ]:
borough_speeds = regular_routes.groupby("borough")["average_speed"].mean()
borough_speeds.sort_values().plot(kind="bar")
plt.title("Average Speed By Borough")
plt.xlabel("Borough")
plt.ylabel("Average Speed (mph)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Visualizations**

In [ ]:
import geopandas as gpd

url_routes = "https://data.ny.gov/resource/bzwk-3hb4.geojson?$where=in_effect='true'&$limit=5000"
routes = gpd.read_file(url_routes)
print(routes.shape)
print(df.shape)
print("Routes dataset sample:",routes["route_id"].head(20).tolist())
print("Speed dataset sample:", regular_routes["route_id"].head(20).tolist())
print("Unique routes in map data:", routes["route_id"].nunique())
print("Unique routes in speed data:", regular_routes["route_id"].nunique())

routes_one = routes.drop_duplicates(subset="route_id")
print(routes_one.shape)

merged = routes_one.merge(route_speeds_clean, on="route_id", how="inner")
print(merged.shape)
print(merged.columns.tolist())

merged["speed_normalized"] = (merged["average_speed"] - merged["average_speed"].min()) / (merged["average_speed"].max() - merged["average_speed"].min())
# Take each speed value, subtract the slowest speed so the minimum becomes zero, then divide by the total range so the maximum becomes one. Every route ends up somewhere between 0 and 1
# for every row, do (this speed - slowest speed) / total range

In [ ]:
import folium
import branca.colormap as cm

m = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles="CartoDB positron",
    min_zoom=10,
    max_bounds=True
)

colormap = cm.LinearColormap(
    colors=["red", "yellow", "green"],
    vmin=merged["average_speed"].min(),
    vmax=merged["average_speed"].max(),
    caption="Average Bus Speed (mph)"
)

for _, row in merged.iterrows():
    feature = {
        "type": "Feature",
        "geometry": row["geometry"].__geo_interface__,
        "properties": {
            "route_id": row["route_id"],
            "route_long_name": row["route_long_name"],
            "average_speed": round(row["average_speed"], 2)
        }
    }
    folium.GeoJson(
        feature,
        style_function=lambda x, speed=row["average_speed"]: {
            "color": colormap(speed),
            "weight": 2,
            "opacity": 0.7
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["route_id", "route_long_name", "average_speed"],
            aliases=["Route:", "Name:", "Avg Speed (mph):"],
            localize=True
        )
    ).add_to(m)

# for every route, package its geometry and data into the format folium expects,
# style it with a color based on its speed, attach a tooltip so hovering shows
# the route info, and add it to the map.

colormap.add_to(m)
m.fit_bounds([[40.4774, -74.2591], [40.9176, -73.7004]])
m

In [65]:
m.save("nyc_bus_speeds_map.html")